# 02 · Sequences: padding, masks, LSTMs, pooling (drill)

**Style B: blind-code drill.** Write each piece from memory, run the check, delete your code.
Reference: `../solutions/drills/02_sequence_models.py`.

**Traces to** `notes/ioai-task-patterns.md` P9 · T25-G-WORDSEG (char vocab with pad 0, padded collate with
lengths, masked BCE, "Embedding + LSTM recommended"). Section 6 writes a recurrent cell by hand.

Total suggested time: **55 min**.

In [ ]:
import re, collections
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from datasets import load_dataset
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

texts = load_dataset("fancyzhx/ag_news", split="train")["text"][:40000]
cnt = collections.Counter(w for t in texts for w in re.findall(r"[a-z]+", t.lower()))
WORDS = [w for w, _ in cnt.most_common(6000) if 3 <= len(w) <= 8][:2500]
def make_compounds(n, seed):
    rng, out = np.random.default_rng(seed), {}
    while len(out) < n:
        parts = [WORDS[i] for i in rng.integers(0, len(WORDS), rng.integers(2, 5))]
        out["".join(parts)] = [b for p in parts for b in [0] * (len(p) - 1) + [1]]
    return list(out.items())
train_data, val_data = make_compounds(30000, 1), make_compounds(3000, 2)

## 1. Vocabulary and encoding ⏱ 5 min

- `build_vocab(words) -> dict`: char → id, ids starting at **1** (0 = padding).
- `encode(word, vocab) -> LongTensor` of shape `(len(word),)`. Unknown chars map to 0.

In [ ]:
vocab = build_vocab([w for w, _ in train_data])
assert min(vocab.values()) == 1 and len(set(vocab.values())) == len(vocab)
e = encode("abc", vocab)
assert e.dtype == torch.long and e.shape == (3,) and (e > 0).all()
assert encode("ab#", vocab)[2].item() == 0
print("section 1 ok")

## 2. Padding collate ⏱ 7 min

`collate(batch)`: `batch` is a list of `(word, labels)` pairs. Return `(ids, y, lengths)`:
`ids` int64 `(B, L)` padded with 0, `y` float32 `(B, L)` padded with 0, `lengths` int64 `(B,)`.
Use the global `vocab`.

In [ ]:
ids, y, lengths = collate([("abcd", [0, 1, 0, 1]), ("ab", [0, 1])])
assert ids.shape == (2, 4) and ids.dtype == torch.long and ids[1, 2:].tolist() == [0, 0]
assert y.dtype == torch.float32 and y[1].tolist() == [0, 1, 0, 0]
assert lengths.tolist() == [4, 2]
print("section 2 ok")

## 3. Padding-invariant BiLSTM tagger ⏱ 12 min

`Segmenter(vocab_size)`: an `nn.Module` whose `forward(ids, lengths)` returns logits `(B, L)`. Use an Embedding
(pad 0) and a bidirectional LSTM. The logits of a word must **not** depend on which other words share its batch.

In [ ]:
m = Segmenter(len(vocab) + 1).to(device).eval()
i1, _, l1 = collate([("house", [0] * 5)])
i2, _, l2 = collate([("house", [0] * 5), ("internationalmarket", [0] * 19)])
with torch.no_grad():
    o1, o2 = m(i1.to(device), l1), m(i2.to(device), l2)
assert o2.shape == (2, 19)
assert torch.allclose(o1[0], o2[0, :5], atol=2e-3), "outputs depend on padding (pack the sequence)"
print("section 3 ok")

## 4. Masked BCE ⏱ 5 min

`masked_bce(logits, y, lengths) -> scalar`: binary cross-entropy with logits, averaged over the **real**
positions only.

In [ ]:
lg = torch.randn(3, 6); yy = torch.randint(0, 2, (3, 6)).float(); ln = torch.tensor([6, 2, 4])
want = torch.cat([F.binary_cross_entropy_with_logits(lg[i, :n], yy[i, :n], reduction="none") for i, n in enumerate(ln)]).mean()
assert torch.isclose(masked_bce(lg, yy, ln), want, atol=1e-6)
lg[1, 3:] = 1e4
assert torch.isclose(masked_bce(lg, yy, ln), want, atol=1e-6), "padded positions leak into the loss"
print("section 4 ok")

## 5. Train to segment-F1 ≥ 0.85 ⏱ 12 min

Train a `Segmenter` called `seg_model` on `train_data`. Then write `predict_boundaries(model, words) -> list[list[int]]`
(one 0/1 list per word, same length as the word). The check scores `val_data` with the T25-G-WORDSEG segment F1.

In [ ]:
def _segments(labels):
    segs, start = set(), 0
    for i, v in enumerate(labels):
        if v == 1: segs.add((start, i + 1)); start = i + 1
    if start < len(labels): segs.add((start, len(labels)))
    return segs
def _f1(g, p):
    g, p = _segments(g), _segments(p); tp = len(g & p)
    pr, rc = (tp / len(p) if p else 0), (tp / len(g) if g else 0)
    return 2 * pr * rc / (pr + rc) if pr + rc else 0.0
words = [w for w, _ in val_data]
preds = predict_boundaries(seg_model, words)
assert len(preds) == len(words) and all(len(p) == len(w) for p, w in zip(preds, words))
score = np.mean([_f1(l, p) for (_, l), p in zip(val_data, preds)])
assert score >= 0.85, score
print(f"section 5 ok (segment F1 {score:.4f})")

## 6. An Elman RNN by hand ⏱ 8 min

`rnn_forward(x, h0, W_ih, W_hh, b_ih, b_hh) -> (outputs, h_last)` for a single-layer tanh RNN:
`h_t = tanh(x_t W_ihᵀ + b_ih + h_{t-1} W_hhᵀ + b_hh)`. Here `x` is `(B, T, D)`, `h0` is `(B, H)`, and outputs is
`(B, T, H)`. It must match `nn.RNN(batch_first=True)`.

In [ ]:
rnn = nn.RNN(5, 7, batch_first=True)
x = torch.randn(3, 4, 5); h0 = torch.randn(1, 3, 7)
ref_out, ref_h = rnn(x, h0)
with torch.no_grad():
    out, h = rnn_forward(x, h0[0], rnn.weight_ih_l0, rnn.weight_hh_l0, rnn.bias_ih_l0, rnn.bias_hh_l0)
assert torch.allclose(out, ref_out, atol=1e-5) and torch.allclose(h, ref_h[0], atol=1e-5)
print("section 6 ok")